# Notebook 06 — LoRA Fine-Tuning

> **阶段**：Stage 3 Train · **预计时间**：40–90 分钟（CPU 冒烟） · **平台**：Kaggle Notebook

LoRA 是资源—性能的权衡方法，不是神奇开关。本 Notebook 让你亲眼看到参数量与代价。


# Learning Objectives

- 理解 Full Fine-Tuning 与 LoRA 的差异（可训练参数、显存、速度）；
- 掌握 r / alpha / dropout / target_modules 的含义；
- 打印并比较 r=4/8/16 的 trainable 参数量；
- 完成 adapter 保存 → 重载 → 推理的最小闭环并记录资源数据。


# Why This Matters

256M 模型全量微调在 T4 16GB 上可行但昂贵；LoRA 把可训练参数降到千分之几，让 Kaggle 免费 GPU 成为训练平台。理解其中的权衡才能正确设计实验。


# Concepts

| | Full Fine-Tuning | LoRA |
| --- | --- | --- |
| 更新对象 | 全部权重 | 低秩增量 A·B（冻结原权重） |
| 可训练参数 | ~100% | 通常 <1% |
| 显存/速度 | 高/慢 | 低/快 |
| 关键超参 | lr、epochs | r、alpha、dropout、target_modules |


## Step 1 — 基线参数量


In [ ]:

# 仓库路径定位：兼容「Notebook 位于仓库根目录」与「仓库克隆在 /kaggle/working 子目录」
from pathlib import Path
import sys

REPO_ROOT = Path.cwd().resolve()
if not (REPO_ROOT / "src" / "model.py").exists():
    matches = [p for p in REPO_ROOT.iterdir() if p.is_dir() and (p / "src" / "model.py").exists()]
    if not matches:
        raise FileNotFoundError(
            "未找到仓库根目录。请按 notebooks/README.md 把仓库克隆到 /kaggle/working，"
            "或把本 Notebook 放在仓库根目录。"
        )
    REPO_ROOT = matches[0].resolve()
sys.path.insert(0, str(REPO_ROOT))
print("REPO_ROOT =", REPO_ROOT)

from src.model import SmolDoclingAdapter
from src.training import parameter_report

adapter = SmolDoclingAdapter().load()
base = parameter_report(adapter.model)
print('base:', base)


## Step 2 — 加上 LoRA（r=8）


In [ ]:

# 仓库路径定位：兼容「Notebook 位于仓库根目录」与「仓库克隆在 /kaggle/working 子目录」
from pathlib import Path
import sys

REPO_ROOT = Path.cwd().resolve()
if not (REPO_ROOT / "src" / "model.py").exists():
    matches = [p for p in REPO_ROOT.iterdir() if p.is_dir() and (p / "src" / "model.py").exists()]
    if not matches:
        raise FileNotFoundError(
            "未找到仓库根目录。请按 notebooks/README.md 把仓库克隆到 /kaggle/working，"
            "或把本 Notebook 放在仓库根目录。"
        )
    REPO_ROOT = matches[0].resolve()
sys.path.insert(0, str(REPO_ROOT))
print("REPO_ROOT =", REPO_ROOT)

from src.training import setup_lora, parameter_report

BASE_MODEL = adapter.model  # 保留原始模型副本，供 Step 5 做 r 对照
adapter.model = setup_lora(adapter.model, r=8, alpha=16, dropout=0.05)
lora = parameter_report(adapter.model)
print('LoRA r=8:', lora)
print('可训练参数减少为全量的 %.4f%%' % (100 * lora['trainable_parameters'] / base['total_parameters']))


## Step 3 — LoRA 冒烟训练（记录显存/时间）


In [ ]:

# 仓库路径定位：兼容「Notebook 位于仓库根目录」与「仓库克隆在 /kaggle/working 子目录」
from pathlib import Path
import sys

REPO_ROOT = Path.cwd().resolve()
if not (REPO_ROOT / "src" / "model.py").exists():
    matches = [p for p in REPO_ROOT.iterdir() if p.is_dir() and (p / "src" / "model.py").exists()]
    if not matches:
        raise FileNotFoundError(
            "未找到仓库根目录。请按 notebooks/README.md 把仓库克隆到 /kaggle/working，"
            "或把本 Notebook 放在仓库根目录。"
        )
    REPO_ROOT = matches[0].resolve()
sys.path.insert(0, str(REPO_ROOT))
print("REPO_ROOT =", REPO_ROOT)

from src import data
from src.prompts import get_prompt
from src.training import SFTDataset, train_sft

data_root = data.find_dataset_root()
annotations = data.load_annotations(data_root)
split = data.build_teaching_split(annotations, n_train=4, n_val=2, seed=42)
records = data.build_sft_records(split['train'], data_root, get_prompt('v0'))
ds = SFTDataset(records, adapter.processor)

result = train_sft(
    adapter.model, ds, epochs=1, lr=2e-4, batch_size=1,
    max_steps=2, device=adapter.device,
    output_dir=REPO_ROOT / 'results' / 'lora' / 'r8_smoke',
)
print(result)


## Step 4 — 保存 adapter → 重载 → 推理


In [ ]:

# 仓库路径定位：兼容「Notebook 位于仓库根目录」与「仓库克隆在 /kaggle/working 子目录」
from pathlib import Path
import sys

REPO_ROOT = Path.cwd().resolve()
if not (REPO_ROOT / "src" / "model.py").exists():
    matches = [p for p in REPO_ROOT.iterdir() if p.is_dir() and (p / "src" / "model.py").exists()]
    if not matches:
        raise FileNotFoundError(
            "未找到仓库根目录。请按 notebooks/README.md 把仓库克隆到 /kaggle/working，"
            "或把本 Notebook 放在仓库根目录。"
        )
    REPO_ROOT = matches[0].resolve()
sys.path.insert(0, str(REPO_ROOT))
print("REPO_ROOT =", REPO_ROOT)

import torch

ckpt = REPO_ROOT / 'results' / 'lora' / 'r8_smoke' / 'adapter.pt'
state = torch.load(ckpt, map_location='cpu')
adapter.model.load_state_dict(state)
print('adapter 已重载:', ckpt, '| 参数键数:', len(state))


## Step 5 — r=4/8/16 参数量对比（不训练）


In [ ]:

# 仓库路径定位：兼容「Notebook 位于仓库根目录」与「仓库克隆在 /kaggle/working 子目录」
from pathlib import Path
import sys

REPO_ROOT = Path.cwd().resolve()
if not (REPO_ROOT / "src" / "model.py").exists():
    matches = [p for p in REPO_ROOT.iterdir() if p.is_dir() and (p / "src" / "model.py").exists()]
    if not matches:
        raise FileNotFoundError(
            "未找到仓库根目录。请按 notebooks/README.md 把仓库克隆到 /kaggle/working，"
            "或把本 Notebook 放在仓库根目录。"
        )
    REPO_ROOT = matches[0].resolve()
sys.path.insert(0, str(REPO_ROOT))
print("REPO_ROOT =", REPO_ROOT)

import copy
from src.training import setup_lora, parameter_report

table = []
for r in (4, 8, 16):
    m = setup_lora(copy.deepcopy(BASE_MODEL), r=r, alpha=2*r)
    rep = parameter_report(m)
    table.append({'r': r, 'trainable': rep['trainable_parameters'], 'trainable_pct': rep['trainable_pct']})
import pandas as pd
display(pd.DataFrame(table))


# What You Should Observe

- r 增大 → 可训练参数线性增长，但训练更慢、显存更高；
- 参数量小 ≠ 一定更差：任务难度决定需要的秩；
- smoke 训练的 2 个 step 无法比较 r 的性能，GPU 上的完整对比才是证据。


# Research Checkpoint

> **为什么说 LoRA 是资源—性能的权衡，而不是一个神奇开关？** 结合你今天看到的可训练参数比例、r 的代价，以及需要验证的三个前提（数据、评测、唯一变量）作答。

**TODO：** 答案写入 `results/nb06/research_checkpoint.md`。


# Exercises

1. **TODO：** 在 GPU 环境完成 r=4/8/16 各一次小训练（同数据、同 step 数），记录 VRAM/时间/val loss，并用 Notebook 07 评测对比；
2. **TODO：** 改变 target_modules（例如只留 q_proj/v_proj），观察可训练参数变化并解释对表达能力的影响；
3. **TODO：** 把 alpha 固定为 2r 之外的取值（如 alpha=r），说明 lora_alpha 缩放机制（lora_alpha/r）对有效学习率的影响。


# Takeaways

- LoRA 让 Kaggle 免费 GPU 训练 256M 模型成为可能；
- 打印参数报告是每个 LoRA 实验的必修动作；
- 超参对比遵循唯一变量原则，且必须回到官方评测。

**下一步**：Phase 4 — [Notebook 08](08_Error_Analysis.ipynb)（错误分析）、[Notebook 09](09_Ablation_Study.ipynb)（消融）与 [Notebook 10](10_From_Experiments_to_Research_Questions.ipynb)。
